In [ ]:
# Import Libraries

import math
import pickle

from tqdm import tqdm
from collections import Counter
from datasets import load_dataset, load_dataset_builder
from transformers import AutoTokenizer

In [ ]:
# Models to Use

# EXAONE
# LGAI-EXAONE/EXAONE-4.0-1.2B

# Kanana
# kakaocorp/kanana-1.5-2.1b-base

# polyglot-ko
# EleutherAI/polyglot-ko-1.3b

In [ ]:
# Settings

MODEL = "EleutherAI/polyglot-ko-1.3b"

_tag      = MODEL.split("/")[-1]
_out_path = f"./results/unigram/unigram_{_tag}_wiki.pkl"

print(f"Model : {MODEL}")
print(f"Output: {_out_path}")


In [ ]:
# Function to Build Unigram

def build_unigram_logpu_from_wikimedia_wikipedia(
    tokenizer,
    wiki_config = "20231101.ko",
    max_docs    = None,
    alpha       = 1.0
):
    ds = load_dataset(
        "wikimedia/wikipedia",
        wiki_config,
        split     = "train",
        streaming = True
    )

    builder    = load_dataset_builder("wikimedia/wikipedia", wiki_config)
    total_docs = builder.info.splits["train"].num_examples
    total_steps = max_docs if max_docs is not None else total_docs

    counter = Counter()
    total   = 0
    special = set(tokenizer.all_special_ids)

    for i, ex in tqdm(
        enumerate(ds),
        total = total_steps,
        desc  = "Building unigram",
        unit  = "doc"
    ):
        if max_docs is not None and i >= max_docs:
            break

        text = (ex.get("text") or ex.get("content") or "").strip()
        if not text:
            continue

        ids = tokenizer.encode(text, add_special_tokens=False)

        if special:
            ids = [tid for tid in ids if tid not in special]

        counter.update(ids)
        total += len(ids)

    if total == 0:
        raise RuntimeError("No tokens counted. Check wiki_config or field names.")

    V     = len(tokenizer)
    denom = total + alpha * V

    log_pu = {
        tid: math.log((counter.get(tid, 0) + alpha) / denom)
        for tid in range(V)
    }

    meta = {
        "model":        MODEL,
        "wiki_dataset": "wikimedia/wikipedia",
        "wiki_config":  wiki_config,
        "max_docs":     max_docs,
        "alpha":        alpha,
        "vocab_size":   V,
        "total_tokens": total,
        "total_docs":   total_docs
        }

    return log_pu, meta

In [ ]:
# Function to Save Unigram

def save_unigram(log_pu, meta, out_path):
    payload = {
        "log_pu": log_pu,
        "meta":   meta
    }

    with open(out_path, "wb") as f:
        pickle.dump(payload, f)

    print(f"[DONE] Saved unigram -> {out_path}")
    print("[META]")

    for k, v in meta.items():
        print(f"  {k}: {v}")

In [ ]:
# Run

tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast = True)

log_pu, meta = build_unigram_logpu_from_wikimedia_wikipedia(
    tokenizer   = tokenizer,
    wiki_config = "20231101.ko",
    max_docs    = None,
    alpha       = 1.0
)

save_unigram(log_pu, meta, out_path = _out_path)